[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/infer-actively/pymdp/blob/main/examples/advanced/infer_states_optimization/methods_test.ipynb)

In [ ]:
import sys
if "google.colab" in sys.modules:
    %pip install "inferactively-pymdp" -q

In [ ]:
import numpy as np
import jax.numpy as jnp
import jax.tree_util as jtu
import jax.experimental.sparse as jsparse
from jax import nn, vmap, jit, block_until_ready
from functools import partial

from pymdp.utils import init_A_and_D_from_spec, get_sample_obs, generate_agent_specs_from_parameter_sets

# Hybrid
from pymdp.utils import apply_padding_batched
from pymdp.maths import compute_log_likelihoods_padded, deconstruct_lls

# Hybrid block
from pymdp.utils import build_block_diag_A, preprocess_A_for_block_diag, prepare_obs_for_block_diag, concatenate_observations_block_diag
from pymdp.maths import compute_log_likelihoods_block_diag, deconstruct_log_likelihoods_block_diag

from pymdp.algos import run_factorized_fpi_hybrid # For hybrid and hybrid block

# End2end padded
from pymdp.utils import apply_A_end2end_padding_batched, apply_obs_end2end_padding_batched
from pymdp.maths import compute_log_likelihood_per_modality_end2end_padded
from pymdp.algos import run_factorized_fpi_end2end_padded

# Clustered hybrid
from pymdp.utils import get_A_dep_clusters, apply_padding_per_cluster
from pymdp.maths import compute_log_likelihoods_per_cluster, deconstruct_log_likelihoods_per_cluster

# Clustered hybrid block
from pymdp.utils import prep_clustered_block_data
from pymdp.maths import compute_log_likelihoods_block_diag_clustered

# Clustered end2end
from pymdp.utils import apply_A_end2end_padding_per_cluster, apply_obs_end2end_padding_per_cluster
from pymdp.maths import compute_log_likelihoods_end2end_per_cluster
from pymdp.algos import run_factorized_fpi_clustered_end2end

In [1]:
# Define coordinated parameter sets
# (num_factors, num_modalities, state_dim_upper_limit, obs_dim_upper_limit, dim_sampling_type, label)
parameter_sets = [
    (5, 5, 5, 5, 'uniform', 'low'),
    (10, 10, 10, 10, 'uniform', 'medium'),
    (25, 25, 25, 25, 'uniform', 'high'),
    # (125, 125, 125, 125, 'uniform', 'extreme'),  # Uncomment to include extreme cases
]

# Generate agent specs without dumping to file
specs = generate_agent_specs_from_parameter_sets(
    parameter_sets,
    num_agents_per_set=1,
    output_file=None  # Don't save to file
)

spec = specs['arbitrary dependencies'][1]
spec

An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.


{'num_factors': 10,
 'num_modalities': 10,
 'num_states': [5, 9, 5, 8, 8, 5, 2, 5, 6, 5],
 'num_obs': [2, 2, 3, 2, 2, 6, 4, 5, 6, 2],
 'A_dependencies': [[0, 4, 6, 7, 9],
  [1, 3, 8],
  [6, 9],
  [3, 8],
  [3],
  [0, 7, 9],
  [0],
  [2],
  [6],
  [5]],
 'metadata': {'num_factors': 'medium',
  'num_modalities': 'medium',
  'state_dim_upper_limit': 'medium',
  'obs_dim_upper_limit': 'medium',
  'dim_sampling_type': 'uniform'}}

In [ ]:
num_iter = 8
batch_size = 4
A_sparsity_level = None # E.g., 0.8 for 80% sparsity

A, D = init_A_and_D_from_spec(
    spec['num_obs'],
    spec['num_states'],
    spec['A_dependencies'],
    A_sparsity_level=A_sparsity_level,
    batch_size=batch_size
)

obs = get_sample_obs(spec['num_obs'], batch_size=batch_size)
o_vec = [nn.one_hot(o, spec['num_obs'][m]) for m, o in enumerate(obs)]

# place where this happens is important!
# o_vec = jtu.tree_map(lambda x: x[-1], o_vec)

### Original method imported directly from PyMDP

In [ ]:
from pymdp.inference import update_posterior_states

infer_states_orig_pymdp = vmap(
    partial(
        update_posterior_states,
        A_dependencies=spec['A_dependencies'],
        num_iter=num_iter,
        method='fpi'
    )
)

In [2]:
qs = infer_states_orig_pymdp(A, None, o_vec, None, D)
[q.shape for q in qs], qs

([(4, 1, 5),
  (4, 1, 9),
  (4, 1, 5),
  (4, 1, 8),
  (4, 1, 8),
  (4, 1, 5),
  (4, 1, 2),
  (4, 1, 5),
  (4, 1, 6),
  (4, 1, 5)],
 [Array([[[0.45187113, 0.02671731, 0.1678715 , 0.27614862, 0.07739149]],
  
         [[0.32997084, 0.15367417, 0.12569936, 0.15770248, 0.23295327]],
  
         [[0.19599514, 0.23715085, 0.19717616, 0.16339767, 0.20628019]],
  
         [[0.13282087, 0.2977195 , 0.15606053, 0.22212943, 0.19126962]]],      dtype=float32),
  Array([[[0.14009844, 0.10671584, 0.10822471, 0.1036902 , 0.10064806,
           0.11968733, 0.12048035, 0.10865847, 0.09179667]],
  
         [[0.11132124, 0.10635602, 0.10483208, 0.12375002, 0.12294658,
           0.10319223, 0.10803859, 0.10658606, 0.11297718]],
  
         [[0.12635784, 0.0976842 , 0.1176099 , 0.1180978 , 0.12710673,
           0.12441596, 0.09231208, 0.09233592, 0.10407965]],
  
         [[0.10652878, 0.11229939, 0.1188262 , 0.11366414, 0.10121107,
           0.11483853, 0.11037278, 0.11457343, 0.10768572]]], dtype=fl

### Hybrid method

In [ ]:
def infer_states_hybrid(obs_padded, A_padded, D, A_shapes, A_dependencies, num_iter):
    lls_padded = compute_log_likelihoods_padded(obs_padded, A_padded)
    log_likelihoods = deconstruct_lls(lls_padded, A_shapes)
    return vmap(partial(run_factorized_fpi_hybrid, A_dependencies=A_dependencies, num_iter=num_iter))(log_likelihoods, D)

In [ ]:
A_padded = apply_padding_batched(A)
A_shapes = [a.shape for a in A]

if A_sparsity_level is not None:
    A_padded = jsparse.BCOO.fromdense(A_padded, n_batch=1)

# obs preprocessing
obs_padded = apply_padding_batched(jtu.tree_map(lambda x: jnp.squeeze(x, 1), o_vec))

In [3]:
qs1 = infer_states_hybrid(obs_padded, A_padded, D, A_shapes, A_dependencies=spec['A_dependencies'], num_iter=num_iter)
[q1.shape for q1 in qs1], [jnp.allclose(q, q1) for q, q1 in zip(qs, qs1)], qs1

([(4, 1, 5),
  (4, 1, 9),
  (4, 1, 5),
  (4, 1, 8),
  (4, 1, 8),
  (4, 1, 5),
  (4, 1, 2),
  (4, 1, 5),
  (4, 1, 6),
  (4, 1, 5)],
 [Array(True, dtype=bool),
  Array(True, dtype=bool),
  Array(True, dtype=bool),
  Array(True, dtype=bool),
  Array(True, dtype=bool),
  Array(True, dtype=bool),
  Array(True, dtype=bool),
  Array(True, dtype=bool),
  Array(True, dtype=bool),
  Array(True, dtype=bool)],
 [Array([[[0.45187113, 0.02671731, 0.1678715 , 0.27614862, 0.07739149]],
  
         [[0.32997084, 0.15367417, 0.12569936, 0.15770248, 0.23295327]],
  
         [[0.19599514, 0.23715085, 0.19717616, 0.16339767, 0.20628019]],
  
         [[0.13282087, 0.2977195 , 0.15606053, 0.22212943, 0.19126962]]],      dtype=float32),
  Array([[[0.14009844, 0.10671584, 0.10822471, 0.1036902 , 0.10064806,
           0.11968733, 0.12048035, 0.10865847, 0.09179667]],
  
         [[0.11132124, 0.10635602, 0.10483208, 0.12375002, 0.12294658,
           0.10319223, 0.10803859, 0.10658606, 0.11297718]],
  
     

In [4]:
# JIT
apply_padding_batched_jit = jit(partial(apply_padding_batched))
infer_states_hybrid_jit = jit(partial(infer_states_hybrid, A_shapes=A_shapes, A_dependencies=spec['A_dependencies'], num_iter=num_iter))
obs_padded = apply_padding_batched_jit(jtu.tree_map(lambda x: jnp.squeeze(x, 1), o_vec))

qs1 = infer_states_hybrid_jit(obs_padded, A_padded, D)
[q1.shape for q1 in qs1], [jnp.allclose(q, q1) for q, q1 in zip(qs, qs1)], qs1

([(4, 1, 5),
  (4, 1, 9),
  (4, 1, 5),
  (4, 1, 8),
  (4, 1, 8),
  (4, 1, 5),
  (4, 1, 2),
  (4, 1, 5),
  (4, 1, 6),
  (4, 1, 5)],
 [Array(True, dtype=bool),
  Array(True, dtype=bool),
  Array(True, dtype=bool),
  Array(True, dtype=bool),
  Array(True, dtype=bool),
  Array(True, dtype=bool),
  Array(True, dtype=bool),
  Array(True, dtype=bool),
  Array(True, dtype=bool),
  Array(True, dtype=bool)],
 [Array([[[0.45187113, 0.02671731, 0.1678715 , 0.27614862, 0.07739149]],
  
         [[0.32997084, 0.15367417, 0.12569936, 0.15770248, 0.23295327]],
  
         [[0.19599514, 0.23715085, 0.19717616, 0.16339767, 0.20628019]],
  
         [[0.13282087, 0.2977195 , 0.15606053, 0.22212943, 0.19126962]]],      dtype=float32),
  Array([[[0.14009844, 0.10671584, 0.10822471, 0.1036902 , 0.10064806,
           0.11968733, 0.12048035, 0.10865847, 0.09179667]],
  
         [[0.11132124, 0.10635602, 0.10483208, 0.12375002, 0.12294658,
           0.10319223, 0.10803859, 0.10658606, 0.11297718]],
  
     

### Clustered Hybrid method

In [ ]:
def infer_states_clustered_hybrid(obs_clusters, A_clusters, D, c2o_mapping, A_shapes, A_dependencies, num_iter):
    ll_clusters = compute_log_likelihoods_per_cluster(obs_clusters, A_clusters)
    log_likelihoods = deconstruct_log_likelihoods_per_cluster(ll_clusters, A_shapes, c2o_mapping)
    return vmap(partial(run_factorized_fpi_hybrid, A_dependencies=A_dependencies, num_iter=num_iter))(log_likelihoods, D)

In [5]:
c2o_mapping = get_A_dep_clusters(spec['A_dependencies'])
A_clusters = apply_padding_per_cluster(A, c2o_mapping)

if A_sparsity_level is not None:
    A_clusters = [jsparse.BCOO.fromdense(a, n_batch=1) for a in A_clusters]

obs_tmp = jtu.tree_map(lambda x: jnp.squeeze(x, 1), o_vec)
obs_clusters = apply_padding_per_cluster(obs_tmp, c2o_mapping)

qs1 = infer_states_clustered_hybrid(obs_clusters, A_clusters, D, c2o_mapping, A_shapes, A_dependencies=spec['A_dependencies'], num_iter=num_iter)
[q1.shape for q1 in qs1], [jnp.allclose(q, q1) for q, q1 in zip(qs, qs1)], qs1

([(4, 1, 5),
  (4, 1, 9),
  (4, 1, 5),
  (4, 1, 8),
  (4, 1, 8),
  (4, 1, 5),
  (4, 1, 2),
  (4, 1, 5),
  (4, 1, 6),
  (4, 1, 5)],
 [Array(True, dtype=bool),
  Array(True, dtype=bool),
  Array(True, dtype=bool),
  Array(True, dtype=bool),
  Array(True, dtype=bool),
  Array(True, dtype=bool),
  Array(True, dtype=bool),
  Array(True, dtype=bool),
  Array(True, dtype=bool),
  Array(True, dtype=bool)],
 [Array([[[0.45187113, 0.02671731, 0.1678715 , 0.27614862, 0.07739149]],
  
         [[0.32997084, 0.15367417, 0.12569936, 0.15770248, 0.23295327]],
  
         [[0.19599514, 0.23715085, 0.19717616, 0.16339767, 0.20628019]],
  
         [[0.13282087, 0.2977195 , 0.15606053, 0.22212943, 0.19126962]]],      dtype=float32),
  Array([[[0.14009844, 0.10671584, 0.10822471, 0.1036902 , 0.10064806,
           0.11968733, 0.12048035, 0.10865847, 0.09179667]],
  
         [[0.11132124, 0.10635602, 0.10483208, 0.12375002, 0.12294658,
           0.10319223, 0.10803859, 0.10658606, 0.11297718]],
  
     

### Hybrid Block method

In [ ]:
# Infer states hybrid block
def infer_states_hybrid_block(obs, A_big, D, state_shapes, cuts, A_dependencies, num_iter, use_einsum=False):
    """Hybrid inference using block diagonal approach for log-likelihood computation."""
    log_likelihoods = compute_log_likelihoods_block_diag(A_big, obs, state_shapes, cuts, use_einsum=use_einsum)
    return vmap(partial(run_factorized_fpi_hybrid, A_dependencies=A_dependencies, num_iter=num_iter))(log_likelihoods, D)

In [ ]:
# Create a copy with moved axes for block diagonal method (don't modify original A)
A_moveaxis = [jnp.moveaxis(a, 1, -1) for a in A]
# Preprocess A matrices for block diagonal approach
A_big, state_shapes, cuts = preprocess_A_for_block_diag(A_moveaxis)

if A_sparsity_level is not None:
    A_big = jsparse.BCOO.fromdense(A_big, n_batch=1)

obs_tmp = jtu.tree_map(lambda x: jnp.squeeze(x, 1), o_vec)
obs_big = concatenate_observations_block_diag(obs_tmp)

In [6]:
qs1 = infer_states_hybrid_block(obs_big, A_big, D, 
    state_shapes=state_shapes, cuts=cuts, A_dependencies=spec['A_dependencies'], 
    num_iter=num_iter, use_einsum=False
)

[q1.shape for q1 in qs1], [jnp.allclose(q, q1) for q, q1 in zip(qs, qs1)], qs1

([(4, 1, 5),
  (4, 1, 9),
  (4, 1, 5),
  (4, 1, 8),
  (4, 1, 8),
  (4, 1, 5),
  (4, 1, 2),
  (4, 1, 5),
  (4, 1, 6),
  (4, 1, 5)],
 [Array(True, dtype=bool),
  Array(True, dtype=bool),
  Array(True, dtype=bool),
  Array(True, dtype=bool),
  Array(True, dtype=bool),
  Array(True, dtype=bool),
  Array(True, dtype=bool),
  Array(True, dtype=bool),
  Array(True, dtype=bool),
  Array(True, dtype=bool)],
 [Array([[[0.45187113, 0.02671731, 0.1678715 , 0.27614862, 0.07739149]],
  
         [[0.32997084, 0.15367417, 0.12569936, 0.15770248, 0.23295327]],
  
         [[0.19599514, 0.23715085, 0.19717616, 0.16339767, 0.20628019]],
  
         [[0.13282087, 0.2977195 , 0.15606053, 0.22212943, 0.19126962]]],      dtype=float32),
  Array([[[0.14009844, 0.10671584, 0.10822471, 0.1036902 , 0.10064806,
           0.11968733, 0.12048035, 0.10865847, 0.09179667]],
  
         [[0.11132124, 0.10635602, 0.10483208, 0.12375002, 0.12294658,
           0.10319223, 0.10803859, 0.10658606, 0.11297718]],
  
     

In [7]:
# JIT
use_einsum=False
concatenate_observations_block_diag_jit = jit(partial(concatenate_observations_block_diag)) # just for obs
infer_states_hybrid_block_jit = jit(partial(infer_states_hybrid_block, state_shapes=state_shapes, cuts=cuts, A_dependencies=spec['A_dependencies'], num_iter=num_iter, use_einsum=use_einsum))
obs_big = concatenate_observations_block_diag_jit(obs_tmp) # add padding of obs before running the infer states

qs1 = infer_states_hybrid_block_jit(obs_big, A_big, D)
[q1.shape for q1 in qs1], [jnp.allclose(q, q1) for q, q1 in zip(qs, qs1)], qs1

([(4, 1, 5),
  (4, 1, 9),
  (4, 1, 5),
  (4, 1, 8),
  (4, 1, 8),
  (4, 1, 5),
  (4, 1, 2),
  (4, 1, 5),
  (4, 1, 6),
  (4, 1, 5)],
 [Array(True, dtype=bool),
  Array(True, dtype=bool),
  Array(True, dtype=bool),
  Array(True, dtype=bool),
  Array(True, dtype=bool),
  Array(True, dtype=bool),
  Array(True, dtype=bool),
  Array(True, dtype=bool),
  Array(True, dtype=bool),
  Array(True, dtype=bool)],
 [Array([[[0.45187113, 0.02671731, 0.1678715 , 0.27614862, 0.07739149]],
  
         [[0.32997084, 0.15367417, 0.12569936, 0.15770248, 0.23295327]],
  
         [[0.19599514, 0.23715085, 0.19717616, 0.16339767, 0.20628019]],
  
         [[0.13282087, 0.2977195 , 0.15606053, 0.22212943, 0.19126962]]],      dtype=float32),
  Array([[[0.14009844, 0.10671584, 0.10822471, 0.1036902 , 0.10064806,
           0.11968733, 0.12048035, 0.10865847, 0.09179667]],
  
         [[0.11132124, 0.10635602, 0.10483208, 0.12375002, 0.12294658,
           0.10319223, 0.10803859, 0.10658606, 0.11297718]],
  
     

### Clustered Hybrid Block method

In [ ]:
def infer_states_clustered_hybrid_block(A_groups, obs_groups, D, shape_groups, cut_groups, group_mapping, A_dependencies, num_iter):
    num_modalities = len(A_dependencies)
    log_likelihoods = compute_log_likelihoods_block_diag_clustered(A_groups, obs_groups, shape_groups, cut_groups, group_mapping, num_modalities)
    return vmap(partial(run_factorized_fpi_hybrid, A_dependencies=A_dependencies, num_iter=num_iter))(log_likelihoods, D)

In [8]:
# Cluster modalities (host-side preprocessing) and build one block-diagonal system per group
A_block, obs_block, state_shapes_block, cuts_block, group_mapping = prep_clustered_block_data(A, obs_tmp)

if A_sparsity_level is not None:
    A_block = [jsparse.BCOO.fromdense(a, n_batch=1) for a in A_block]

qs1 = infer_states_clustered_hybrid_block(A_block, obs_block, D, state_shapes_block, cuts_block, group_mapping, A_dependencies=spec['A_dependencies'], num_iter=num_iter)
[q1.shape for q1 in qs1], [jnp.allclose(q, q1) for q, q1 in zip(qs, qs1)], qs1

([(4, 1, 5),
  (4, 1, 9),
  (4, 1, 5),
  (4, 1, 8),
  (4, 1, 8),
  (4, 1, 5),
  (4, 1, 2),
  (4, 1, 5),
  (4, 1, 6),
  (4, 1, 5)],
 [Array(True, dtype=bool),
  Array(True, dtype=bool),
  Array(True, dtype=bool),
  Array(True, dtype=bool),
  Array(True, dtype=bool),
  Array(True, dtype=bool),
  Array(True, dtype=bool),
  Array(True, dtype=bool),
  Array(True, dtype=bool),
  Array(True, dtype=bool)],
 [Array([[[0.45187113, 0.02671731, 0.1678715 , 0.27614862, 0.07739149]],
  
         [[0.32997084, 0.15367417, 0.12569936, 0.15770248, 0.23295327]],
  
         [[0.19599514, 0.23715085, 0.19717616, 0.16339767, 0.20628019]],
  
         [[0.13282087, 0.2977195 , 0.15606053, 0.22212943, 0.19126962]]],      dtype=float32),
  Array([[[0.14009844, 0.10671584, 0.10822471, 0.1036902 , 0.10064806,
           0.11968733, 0.12048035, 0.10865847, 0.09179667]],
  
         [[0.11132124, 0.10635602, 0.10483208, 0.12375002, 0.12294658,
           0.10319223, 0.10803859, 0.10658606, 0.11297718]],
  
     

### End2End padded method

In [ ]:
def infer_states_end2end_padded(A_padded, obs_padded, D, A_dependencies, max_obs_dim, max_state_dim, num_iter, sparsity=None):
    lls_padded = compute_log_likelihood_per_modality_end2end_padded(obs_padded, A_padded, sparsity=sparsity)
    return run_factorized_fpi_end2end_padded(lls_padded, D, A_dependencies, max_obs_dim, max_state_dim, num_iter)

In [ ]:
A_padded = apply_A_end2end_padding_batched(A)

if A_sparsity_level is not None:
    A_padded = jsparse.BCOO.fromdense(A_padded)

max_obs_dim = A_padded.shape[2]
max_state_dim = max(A_padded.shape[3:])

# obs preprocessing
obs_padded = apply_obs_end2end_padding_batched(jtu.tree_map(lambda x: jnp.squeeze(x, 1), o_vec), max_obs_dim)

In [9]:
qs1 = infer_states_end2end_padded(A_padded, obs_padded, D, spec['A_dependencies'], max_obs_dim, max_state_dim, num_iter, sparsity='ll_only')

[q1.shape for q1 in qs1], [jnp.allclose(q, q1) for q, q1 in zip(qs, qs1)], qs1

([(4, 1, 5),
  (4, 1, 9),
  (4, 1, 5),
  (4, 1, 8),
  (4, 1, 8),
  (4, 1, 5),
  (4, 1, 2),
  (4, 1, 5),
  (4, 1, 6),
  (4, 1, 5)],
 [Array(True, dtype=bool),
  Array(True, dtype=bool),
  Array(True, dtype=bool),
  Array(True, dtype=bool),
  Array(True, dtype=bool),
  Array(True, dtype=bool),
  Array(True, dtype=bool),
  Array(True, dtype=bool),
  Array(True, dtype=bool),
  Array(True, dtype=bool)],
 [Array([[[0.4518712 , 0.02671732, 0.16787152, 0.2761485 , 0.0773915 ]],
  
         [[0.32997087, 0.15367419, 0.12569931, 0.15770242, 0.23295328]],
  
         [[0.19599503, 0.23715094, 0.19717613, 0.16339773, 0.20628026]],
  
         [[0.13282083, 0.29771954, 0.1560606 , 0.22212934, 0.19126965]]],      dtype=float32),
  Array([[[0.1400984 , 0.10671582, 0.1082247 , 0.1036902 , 0.10064805,
           0.11968729, 0.12048034, 0.10865844, 0.09179666]],
  
         [[0.11132124, 0.10635601, 0.10483208, 0.12375006, 0.12294656,
           0.10319225, 0.1080386 , 0.10658605, 0.11297718]],
  
     

In [10]:
# JIT
apply_obs_padding_batched_jit = jit(partial(apply_obs_end2end_padding_batched, max_obs_dim=max_obs_dim))
infer_states_partially_padded_jit = jit(partial(infer_states_end2end_padded, A_dependencies=spec['A_dependencies'], max_obs_dim=max_obs_dim, max_state_dim=max_state_dim, num_iter=num_iter, sparsity='ll_only'))
obs_padded = apply_obs_padding_batched_jit(jtu.tree_map(lambda x: jnp.squeeze(x, 1), o_vec))

qs1 = infer_states_partially_padded_jit(A_padded, obs_padded, D)
[q1.shape for q1 in qs1], [jnp.allclose(q, q1) for q, q1 in zip(qs, qs1)], qs1

([(4, 1, 5),
  (4, 1, 9),
  (4, 1, 5),
  (4, 1, 8),
  (4, 1, 8),
  (4, 1, 5),
  (4, 1, 2),
  (4, 1, 5),
  (4, 1, 6),
  (4, 1, 5)],
 [Array(True, dtype=bool),
  Array(True, dtype=bool),
  Array(True, dtype=bool),
  Array(True, dtype=bool),
  Array(True, dtype=bool),
  Array(True, dtype=bool),
  Array(True, dtype=bool),
  Array(True, dtype=bool),
  Array(True, dtype=bool),
  Array(True, dtype=bool)],
 [Array([[[0.4518712 , 0.02671732, 0.16787152, 0.2761485 , 0.0773915 ]],
  
         [[0.32997087, 0.15367419, 0.12569931, 0.15770242, 0.23295328]],
  
         [[0.19599503, 0.23715094, 0.19717613, 0.16339773, 0.20628026]],
  
         [[0.13282083, 0.29771954, 0.1560606 , 0.22212934, 0.19126965]]],      dtype=float32),
  Array([[[0.1400984 , 0.10671582, 0.1082247 , 0.1036902 , 0.10064805,
           0.11968729, 0.12048034, 0.10865844, 0.09179666]],
  
         [[0.11132124, 0.10635601, 0.10483208, 0.12375006, 0.12294656,
           0.10319225, 0.1080386 , 0.10658605, 0.11297718]],
  
     

### Clustered End2End method

In [ ]:
def infer_states_clustered_end2end(obs_clusters, A_clusters, D, c2o_mapping, c2s_mapping, max_state_dims, A_dependencies, num_iter, sparsity=None):
    ll_clusters = compute_log_likelihoods_end2end_per_cluster(obs_clusters, A_clusters, sparsity)
    return run_factorized_fpi_clustered_end2end(ll_clusters, D, c2o_mapping, c2s_mapping, max_state_dims, A_dependencies, num_iter)

In [11]:
A_clusters = apply_A_end2end_padding_per_cluster(A, c2o_mapping)
max_obs_dims = [a.shape[2] for a in A_clusters]
max_state_dims = [a.shape[-1] for a in A_clusters]
obs_clusters = apply_obs_end2end_padding_per_cluster(obs_tmp, c2o_mapping, max_obs_dims)
c2s_mapping = [[s for o in o_list for s in spec['A_dependencies'][o]] for o_list in c2o_mapping]

if A_sparsity_level is not None:
    A_clusters = [jsparse.BCOO.fromdense(a) for a in A_clusters]

qs1 = infer_states_clustered_end2end(obs_clusters, A_clusters, D, c2o_mapping, c2s_mapping, max_state_dims, A_dependencies=spec['A_dependencies'], num_iter=num_iter, sparsity='ll_only')
[q1.shape for q1 in qs1], [jnp.allclose(q, q1) for q, q1 in zip(qs, qs1)], qs1

([(4, 1, 5),
  (4, 1, 9),
  (4, 1, 5),
  (4, 1, 8),
  (4, 1, 8),
  (4, 1, 5),
  (4, 1, 2),
  (4, 1, 5),
  (4, 1, 6),
  (4, 1, 5)],
 [Array(True, dtype=bool),
  Array(True, dtype=bool),
  Array(True, dtype=bool),
  Array(True, dtype=bool),
  Array(True, dtype=bool),
  Array(True, dtype=bool),
  Array(True, dtype=bool),
  Array(True, dtype=bool),
  Array(True, dtype=bool),
  Array(True, dtype=bool)],
 [Array([[[0.45187113, 0.02671732, 0.1678715 , 0.27614862, 0.07739149]],
  
         [[0.3299709 , 0.1536742 , 0.12569931, 0.15770242, 0.23295319]],
  
         [[0.19599512, 0.23715095, 0.19717604, 0.16339767, 0.20628019]],
  
         [[0.13282083, 0.29771954, 0.15606068, 0.22212934, 0.19126965]]],      dtype=float32),
  Array([[[0.1400984 , 0.10671582, 0.1082247 , 0.1036902 , 0.10064805,
           0.11968729, 0.12048034, 0.10865844, 0.09179666]],
  
         [[0.11132124, 0.10635601, 0.10483208, 0.12375006, 0.12294656,
           0.10319225, 0.1080386 , 0.10658605, 0.11297718]],
  
     